 SDXL_Research
Caption, train, compare and generate SDXL LoRAs on the Colab free T4.

1. **Runtime > Change runtime type > T4 GPU**
2. Add your Hugging Face token in **Secrets** (key icon) as `HF_TOKEN` (needed for private repos)
3. Run **Setup** once per session; re-runs skip finished steps.

In [ ]:
#@title Setup: repo + venv (idempotent)
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                   env={**os.environ, "PYTHONPATH": str(REPO / "src")})
print(r.stdout[-3000:])
assert r.returncode == 0, "setup failed (see above)"
nb.hf_token()

# Dataset
Puts images (and any existing `.txt` captions) into one folder.
`test` downloads 5 public dog photos to try the pipeline end to end.

In [ ]:
#@title Get dataset
SOURCE = "test"  #@param ["test", "hf", "drive"]
DATASET_DIR = "/content/dataset"  #@param {type:"string"}
#@markdown **hf**: private/public repo + zip filename
HF_REPO = ""  #@param {type:"string"}
HF_FILE = "dataset.zip"  #@param {type:"string"}
#@markdown **drive**: folder or .zip in your Drive
DRIVE_PATH = "/content/drive/MyDrive/datasets/my_images"  #@param {type:"string"}

args = ["-m", "sdxlr.data", "--dest", DATASET_DIR]
if SOURCE == "test":
    args += ["--source", "test"]
elif SOURCE == "hf":
    args += ["--source", "hf", "--repo", HF_REPO, "--file", HF_FILE]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    args += ["--source", "dir", "--path", DRIVE_PATH]
nb.run(args)

# Caption
Writes one `.txt` per image. Captions describe clothing, pose, background and
lighting, and skip face and body, so the LoRA learns those from the pixels.
Check the first few with `PREVIEW = 3`, then set `PREVIEW = 0` for all.

In [ ]:
#@title Caption (Qwen3-VL / JoyCaption)
MODEL = "qwen"  #@param ["qwen", "qwen-2b", "joy"]
TRIGGER = "ohwx_person"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
PROMPT = ""  #@param {type:"string"}
#@markdown blank PROMPT = identity preset; `{cls}` in a custom prompt = CLASS_WORD
PREVIEW = 3  #@param {type:"integer"}
OVERWRITE = False  #@param {type:"boolean"}

args = ["-m", "sdxlr.caption", "--images", DATASET_DIR, "--model", MODEL,
        "--trigger", TRIGGER, "--class-word", CLASS_WORD, "--limit", PREVIEW]
if PROMPT: args += ["--prompt", PROMPT]
if OVERWRITE: args += ["--overwrite"]
nb.run(args)

# Train
Defaults: `configs/train/sdxl_person.yaml` (proven T4 settings).
With `HF_REPO` set, LoRAs + resume state upload to that **private** repo each
epoch; tick `RESUME` after a disconnect to continue.

In [ ]:
#@title Train SDXL LoRA
RUN_NAME = "test_run"  #@param {type:"string"}
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0/sd_xl_base_1.0.safetensors"  #@param {type:"string"}
#@markdown org/repo/file on HF, an https URL, or a local path
HF_REPO = ""  #@param {type:"string"}
RESUME = False  #@param {type:"boolean"}
EPOCHS = 15  #@param {type:"integer"}
REPEATS = 6  #@param {type:"integer"}
BATCH = 3  #@param {type:"integer"}
LORA_TYPE = "LoRA"  #@param ["LoRA", "LoCon"]
DIM = 24  #@param {type:"integer"}
ALPHA = 16  #@param {type:"integer"}
UNET_LR = 2e-4  #@param {type:"number"}
TE_LR = 2e-5  #@param {type:"number"}
EXTRA = ""  #@param {type:"string"}
#@markdown EXTRA: more overrides as `key=value key=value` (see the yaml)

args = ["-m", "sdxlr.train", "--name", RUN_NAME, "--images", DATASET_DIR, "--model", BASE_MODEL,
        "--set", f"epochs={EPOCHS}", f"repeats={REPEATS}", f"batch_size={BATCH}",
        f"lora_type={LORA_TYPE}", f"network_dim={DIM}", f"network_alpha={ALPHA}",
        f"unet_lr={UNET_LR}", f"text_encoder_lr={TE_LR}", *EXTRA.split()]
if HF_REPO: args += ["--hf-repo", HF_REPO]
if RESUME: args += ["--resume"]
nb.run(args)